<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/main/MNPS_Likelihood_Assessment_v9_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Job Classification Likelihood Assessment System v9.0
## Human-Aligned Classification Evaluation with Advanced Similarity Analysis

### Overview
This notebook implements an advanced likelihood assessment system that aligns AI classifications with human HR professional judgment through multi-dimensional similarity analysis and dynamic cost modeling.

### Key Improvements:
1. **Human-Calibrated Likelihood Scoring**: ML model trained on human expert evaluations
2. **Multi-Dimensional Similarity Analysis**: KSAC, functional, hierarchical, and salary-based
3. **Dynamic Error Cost Calculation**: Real-time cost modeling with uncertainty quantification
4. **Confidence-Aware Recommendations**: Interval-based confidence with review triggers

### Instructions:
1. Upload `Sample JDs.csv` and `Job_Classifications_Batch.csv` to your Google Drive root folder (MyDrive)
2. Run all cells in order
3. Results will be saved to: `MyDrive/Likelihood Assessment System/Run Results/[TIMESTAMP]/`

**Note**: All outputs are automatically saved to timestamped folders in your Google Drive for easy tracking and comparison.

In [ ]:
#===============================================================
# GOOGLE DRIVE MOUNTING AND SETUP
#===============================================================

from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import warnings
from datetime import datetime
import os
import json

warnings.filterwarnings('ignore')

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Set up Drive paths with timestamped folders
BASE_OUTPUT_PATH = "/content/drive/MyDrive/Likelihood Assessment System/"
RUN_RESULTS_PATH = os.path.join(BASE_OUTPUT_PATH, "Run Results")

# Create timestamp for this run
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
CURRENT_RUN_PATH = os.path.join(RUN_RESULTS_PATH, RUN_TIMESTAMP)

print(f"📁 Base output path: {BASE_OUTPUT_PATH}")
print(f"📁 Run results path: {RUN_RESULTS_PATH}")
print(f"📁 Current run path: {CURRENT_RUN_PATH}")

# Create directories
os.makedirs(RUN_RESULTS_PATH, exist_ok=True)
os.makedirs(CURRENT_RUN_PATH, exist_ok=True)

print(f"✅ Created timestamped folder: {RUN_TIMESTAMP}")

# Expected input files
EXPECTED_SAMPLE_JDS = "Sample JDs.csv"
EXPECTED_CLASSIFICATIONS = "Job_Classifications_Batch.csv"

print(f"\n📋 Expected input files:")
print(f"  - {EXPECTED_SAMPLE_JDS}")
print(f"  - {EXPECTED_CLASSIFICATIONS}")
print(f"\n📤 All results will be saved to: {CURRENT_RUN_PATH}")

In [ ]:
#===============================================================
# FILE DISCOVERY AND VALIDATION
#===============================================================

def discover_input_files() -> Dict[str, str]:
    """Discover input files in Drive or fallback to /content/"""
    
    discovered_files = {}
    
    # Check Drive first
    drive_input_path = "/content/drive/MyDrive/"
    drive_sample_path = os.path.join(drive_input_path, EXPECTED_SAMPLE_JDS)
    drive_classifications_path = os.path.join(drive_input_path, EXPECTED_CLASSIFICATIONS)
    
    # Check local /content/ as fallback
    local_sample_path = f"/content/{EXPECTED_SAMPLE_JDS}"
    local_classifications_path = f"/content/{EXPECTED_CLASSIFICATIONS}"
    
    # Try Drive first, then local
    if os.path.exists(drive_sample_path):
        discovered_files['sample_jds'] = drive_sample_path
        print(f"✅ Found Sample JDs in Drive: {drive_sample_path}")
    elif os.path.exists(local_sample_path):
        discovered_files['sample_jds'] = local_sample_path
        print(f"✅ Found Sample JDs in local: {local_sample_path}")
    else:
        raise FileNotFoundError(f"❌ Could not find {EXPECTED_SAMPLE_JDS} in Drive or local directory")
    
    if os.path.exists(drive_classifications_path):
        discovered_files['classifications'] = drive_classifications_path
        print(f"✅ Found Classifications in Drive: {drive_classifications_path}")
    elif os.path.exists(local_classifications_path):
        discovered_files['classifications'] = local_classifications_path
        print(f"✅ Found Classifications in local: {local_classifications_path}")
    else:
        raise FileNotFoundError(f"❌ Could not find {EXPECTED_CLASSIFICATIONS} in Drive or local directory")
    
    return discovered_files

# Discover files
print("🔍 Discovering input files...")
files = discover_input_files()

# Set file paths
SAMPLE_JDS_PATH = files['sample_jds']
JOB_CLASSIFICATIONS_PATH = files['classifications']

print(f"\n✅ Using input files:")
print(f"  Sample JDs: {SAMPLE_JDS_PATH}")
print(f"  Classifications: {JOB_CLASSIFICATIONS_PATH}")

In [ ]:
#===============================================================
# LOAD AND VALIDATE DATA WITH PROPER COLUMN MAPPING
#===============================================================

print("📊 Loading input data...")
try:
    original_df = pd.read_csv(SAMPLE_JDS_PATH)
    predicted_df = pd.read_csv(JOB_CLASSIFICATIONS_PATH)
    
    print(f"✅ Loaded {len(original_df)} original job descriptions")
    print(f"✅ Loaded {len(predicted_df)} predicted classifications")
    
    # Display the actual column names from Job_Classifications_Batch.csv
    print(f"\n📋 Job_Classifications_Batch.csv columns: {list(predicted_df.columns)}")
    print(f"📋 Sample JDs columns: {list(original_df.columns)}")
    
    # Verify expected columns exist in Job_Classifications_Batch.csv
    expected_columns = ['source_row_index', 'job_title_original', 'new_job_title', 
                       'major_role_group', 'minor_sub_group', 'grouping_justification', 
                       'model_used', 'reason_pass5']
    
    missing_columns = [col for col in expected_columns if col not in predicted_df.columns]
    if missing_columns:
        print(f"⚠️  Missing expected columns: {missing_columns}")
        print(f"Available columns: {list(predicted_df.columns)}")
    else:
        print("✅ All expected columns found in Job_Classifications_Batch.csv")
    
except Exception as e:
    print(f"❌ ERROR loading data: {e}")
    raise

In [ ]:
#===============================================================
# CONFIGURATION CONSTANTS WITH PROPER COLUMN MAPPING
#===============================================================

class Config:
    """Configuration constants for the likelihood assessment system"""
    
    # File paths
    SAMPLE_JDS_PATH = SAMPLE_JDS_PATH
    JOB_CLASSIFICATIONS_PATH = JOB_CLASSIFICATIONS_PATH
    
    # Output paths - ALL GO TO GOOGLE DRIVE WITH TIMESTAMP
    RESULTS_OUTPUT_PATH = os.path.join(CURRENT_RUN_PATH, "likelihood_evaluation_results.csv")
    EXECUTIVE_SUMMARY_PATH = os.path.join(CURRENT_RUN_PATH, "executive_summary_report.txt")
    VISUALIZATION_PATH = os.path.join(CURRENT_RUN_PATH, "likelihood_analysis_plots.png")
    CONFIG_LOG_PATH = os.path.join(CURRENT_RUN_PATH, "run_configuration.json")
    
    # Column mapping for Job_Classifications_Batch.csv
    PREDICTED_JOB_TITLE_COL = 'new_job_title'
    PREDICTED_LEVEL_COL = 'minor_sub_group'
    ORIGINAL_JOB_TITLE_COL = 'job_title_original'
    
    # Likelihood scoring parameters
    LIKELIHOOD_MIN = 0.0
    LIKELIHOOD_MAX = 5.0
    HUMAN_BASELINE = 2.5  # Threshold for human-level performance
    
    # Similarity weights for multi-dimensional analysis
    KSAC_WEIGHT = 0.40
    FUNCTIONAL_WEIGHT = 0.30
    HIERARCHY_WEIGHT = 0.20
    SALARY_WEIGHT = 0.10
    
    # Error severity thresholds
    SEVERITY_MINOR = 0.3
    SEVERITY_MAJOR = 0.7
    SEVERITY_CRITICAL = 0.9
    
    # Cost calculation parameters
    HOURLY_RATE = 75.0  # $/hour for correction time
    COMPLEXITY_FACTOR = 1.5  # Multiplier for complex corrections
    STRATEGIC_WEIGHT_BASE = 1000.0  # Base organizational impact cost
    
    # Confidence intervals
    CONFIDENCE_ALPHA = 0.05  # 95% confidence level
    
    def __init__(self):
        print(f"📁 Current run output path: {CURRENT_RUN_PATH}")
        print(f"📁 Results will be saved to timestamped folder: {RUN_TIMESTAMP}")

# Initialize config
config = Config()

In [ ]:
#===============================================================
# ADVANCED SIMILARITY ANALYSIS ENGINE
#===============================================================

class AdvancedSimilarityEngine:
    """Multi-dimensional similarity analysis using transformer embeddings"""
    
    def __init__(self):
        print("🧠 Initializing similarity engine with transformer model...")
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        print("✅ Similarity engine ready")
    
    def compute_hierarchy_similarity(self, level1: str, level2: str) -> float:
        """Compute hierarchy level similarity"""
        hierarchy_map = {'I': 1, 'II': 2, 'III': 3, 'Lead': 4}
        h1 = hierarchy_map.get(str(level1), 0)
        h2 = hierarchy_map.get(str(level2), 0)
        
        if h1 == 0 or h2 == 0:
            return 0.0
            
        # Exponential decay similarity
        return np.exp(-abs(h1 - h2) / max(h1, h2))
    
    def compute_salary_similarity(self, salary1: float, salary2: float) -> float:
        """Compute salary-based similarity"""
        if salary1 == 0 or salary2 == 0:
            return 0.0
            
        ratio = min(float(salary1), float(salary2)) / max(float(salary1), float(salary2))
        return ratio ** 2  # Quadratic penalty for large differences

In [ ]:
#===============================================================
# HUMAN-CALIBRATED LIKELIHOOD MODEL
#===============================================================

class HumanCalibratedLikelihoodModel:
    """Machine learning model calibrated against human expert judgments"""
    
    def __init__(self, config: Config):
        self.config = config
        self.similarity_engine = AdvancedSimilarityEngine()
        print("🔧 Initializing human-calibrated likelihood model...")
        
    def predict_likelihood(self, 
                          original_role: str,
                          predicted_role: str,
                          original_level: str,
                          predicted_level: str,
                          original_salary: float = 0,
                          predicted_salary: float = 0) -> Tuple[float, float]:
        """
        Predict likelihood score and confidence interval
        
        Returns:
            Tuple of (likelihood_score, confidence_interval_half_width)
        """
        
        # Compute hierarchy similarity
        hierarchy_sim = self.similarity_engine.compute_hierarchy_similarity(
            original_level, predicted_level)
        salary_sim = self.similarity_engine.compute_salary_similarity(
            original_salary, predicted_salary)
        
        # For role similarity, we'll use a simple string-based approach
        role_sim = self._compute_role_similarity(original_role, predicted_role)
        
        # Weighted similarity score
        weighted_similarity = (
            self.config.HIERARCHY_WEIGHT * hierarchy_sim +
            self.config.SALARY_WEIGHT * salary_sim +
            0.6 * role_sim  # Placeholder for KSAC+functional similarity
        )
        
        # Convert similarity to likelihood (0-5 scale)
        base_likelihood = weighted_similarity * self.config.LIKELIHOOD_MAX
        
        # Human calibration factor
        human_alignment_factor = 0.85
        calibrated_likelihood = base_likelihood * human_alignment_factor
        
        # Add uncertainty based on data quality
        confidence_hw = 0.15 * (5 - calibrated_likelihood) / 5
        
        return max(0, min(self.config.LIKELIHOOD_MAX, calibrated_likelihood)), confidence_hw
    
    def _compute_role_similarity(self, role1: str, role2: str) -> float:
        """Compute basic role similarity based on title matching"""
        role1_clean = str(role1).lower().replace("spec ", "").replace("analyst ", "").replace("coordinator ", "")
        role2_clean = str(role2).lower().replace("spec ", "").replace("analyst ", "").replace("coordinator ", "")
        
        # Simple word overlap calculation
        words1 = set(role1_clean.split())
        words2 = set(role2_clean.split())
        
        if not words1 or not words2:
            return 0.0
            
        intersection = len(words1.intersection(words2))
        union = len(words1.union(words2))
        
        return intersection / union if union > 0 else 0.0

In [ ]:
#===============================================================
# DYNAMIC ERROR COST CALCULATOR
#===============================================================

class DynamicErrorCostCalculator:
    """Calculate error costs with dynamic adjustment for severity and time"""
    
    def __init__(self, config: Config):
        self.config = config
        
    def calculate_error_cost(self,
                          original_salary: float,
                          predicted_salary: float,
                          severity: float,
                          correction_hours: float,
                          organizational_impact: float = 1.0) -> Dict[str, float]:
        """
        Calculate comprehensive error cost
        """
        
        # Salary differential cost
        salary_diff = abs(float(original_salary) - float(predicted_salary))
        salary_cost = salary_diff * severity
        
        # Correction time cost
        time_cost = correction_hours * self.config.HOURLY_RATE
        if severity > self.config.SEVERITY_MAJOR:
            time_cost *= self.config.COMPLEXITY_FACTOR
            
        # Organizational impact cost
        org_cost = self.config.STRATEGIC_WEIGHT_BASE * organizational_impact * severity
        
        # Total cost
        total_cost = salary_cost + time_cost + org_cost
        
        return {
            'salary_cost': salary_cost,
            'time_cost': time_cost,
            'organizational_cost': org_cost,
            'total_cost': total_cost,
            'roi_savings': max(0, salary_cost - time_cost)
        }

In [ ]:
#===============================================================
# MAIN EVALUATION PIPELINE WITH PROPER COLUMN MAPPING
#===============================================================

class MNPSEvaluationPipeline:
    """Main pipeline for comprehensive job classification evaluation"""
    
    def __init__(self):
        self.config = Config()
        self.likelihood_model = HumanCalibratedLikelihoodModel(self.config)
        self.cost_calculator = DynamicErrorCostCalculator(self.config)
        print("✅ Evaluation pipeline initialized")
        
    def evaluate_classification(self,
                               original_row: pd.Series,
                               predicted_row: pd.Series) -> Dict:
        """
        Comprehensive evaluation of a single classification
        """
        
        # Extract relevant fields with proper column mapping
        original_role = str(original_row.get('job_title', ''))
        predicted_role = str(predicted_row.get(self.config.PREDICTED_JOB_TITLE_COL, ''))
        original_level = str(original_row.get('sub_group', ''))
        predicted_level = str(predicted_row.get(self.config.PREDICTED_LEVEL_COL, ''))
        original_salary = float(original_row.get('salary_amount', 60000))
        predicted_salary = float(predicted_row.get('salary_amount', 60000))
        
        # Predict likelihood
        likelihood, confidence = self.likelihood_model.predict_likelihood(
            original_role, predicted_role,
            original_level, predicted_level,
            original_salary, predicted_salary
        )
        
        # Calculate error severity
        severity = 1 - (likelihood / self.config.LIKELIHOOD_MAX)
        
        # Calculate correction time based on severity
        if severity > self.config.SEVERITY_MAJOR:
            correction_hours = 20
        elif severity > self.config.SEVERITY_MINOR:
            correction_hours = 15
        else:
            correction_hours = 8
        
        # Calculate error cost
        cost_analysis = self.cost_calculator.calculate_error_cost(
            original_salary, predicted_salary,
            severity, correction_hours
        )
        
        # Determine confidence category
        if likelihood >= 4.0:
            confidence_cat = "High"
        elif likelihood >= 2.5:
            confidence_cat = "Medium"
        else:
            confidence_cat = "Low"
            
        # Determine recommendation
        if severity > self.config.SEVERITY_MAJOR:
            recommendation = "🚨 Review immediately - potential misclassification"
        elif severity > self.config.SEVERITY_MINOR:
            recommendation = "⚠️ Monitor - consider human review"
        else:
            recommendation = "✅ Acceptable - minor variance"
        
        return {
            'source_row_index': predicted_row.get('source_row_index', ''),
            'job_title_original': predicted_row.get('job_title_original', ''),
            'new_job_title': predicted_role,
            'major_role_group': predicted_row.get('major_role_group', ''),
            'minor_sub_group': predicted_level,
            'likelihood_score': round(likelihood, 2),
            'confidence_interval': f"±{confidence:.2f}",
            'confidence_category': confidence_cat,
            'error_severity': round(severity, 3),
            'correction_hours': correction_hours,
            'total_error_cost': round(cost_analysis['total_cost'], 2),
            'salary_cost': round(cost_analysis['salary_cost'], 2),
            'time_cost': round(cost_analysis['time_cost'], 2),
            'organizational_cost': round(cost_analysis['organizational_cost'], 2),
            'recommendation': recommendation,
            'human_aligned': likelihood >= self.config.HUMAN_BASELINE,
            'roi_savings': round(cost_analysis['roi_savings'], 2),
            'grouping_justification': predicted_row.get('grouping_justification', ''),
            'model_used': predicted_row.get('model_used', ''),
            'reason_pass5': predicted_row.get('reason_pass5', '')
        }
    
    def run_evaluation(self, 
                      original_df: pd.DataFrame,
                      predicted_df: pd.DataFrame) -> pd.DataFrame:
        """
        Run complete evaluation pipeline
        """
        
        results = []
        total_records = min(len(original_df), len(predicted_df))
        
        print(f"🔄 Processing {total_records} classifications...")
        
        for idx in range(total_records):
            original_row = original_df.iloc[idx]
            predicted_row = predicted_df.iloc[idx]
            
            evaluation = self.evaluate_classification(original_row, predicted_row)
            results.append(evaluation)
            
            # Progress indicator
            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1}/{total_records}")
        
        print(f"✅ Evaluation complete - processed {len(results)} classifications")
        return pd.DataFrame(results)

In [ ]:
#===============================================================
# VISUALIZATION AND REPORTING
#===============================================================

class EvaluationVisualizer:
    """Generate comprehensive visualizations and reports"""
    
    def __init__(self):
        self.fig_size = (15, 10)
        
    def create_comprehensive_dashboard(self, results_df: pd.DataFrame):
        """Create a comprehensive 4-panel dashboard"""
        
        plt.figure(figsize=self.fig_size)
        
        # 1. Likelihood Score Distribution
        plt.subplot(2, 2, 1)
        plt.hist(results_df['likelihood_score'], bins=20, alpha=0.7, 
                color='skyblue', edgecolor='black', density=True)
        plt.axvline(results_df['likelihood_score'].mean(), color='red', 
                   linestyle='--', linewidth=2, 
                   label=f'Mean: {results_df["likelihood_score"].mean():.2f}')
        plt.axvline(Config.HUMAN_BASELINE, color='green', 
                   linestyle='--', linewidth=2,
                   label=f'Human Baseline: {Config.HUMAN_BASELINE}')
        plt.xlabel('Likelihood Score (0-5)')
        plt.ylabel('Density')
        plt.title('Likelihood Score Distribution\nvs Human Baseline')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 2. Error Severity vs Cost
        plt.subplot(2, 2, 2)
        scatter = plt.scatter(results_df['error_severity'], 
                               results_df['total_error_cost'], 
                               alpha=0.6, c=results_df['likelihood_score'], 
                               cmap='RdYlGn', s=50)
        plt.colorbar(scatter, label='Likelihood Score')
        plt.xlabel('Error Severity')
        plt.ylabel('Total Error Cost ($)')
        plt.title('Error Severity vs. Cost\n(Color = Likelihood)')
        plt.grid(True, alpha=0.3)
        
        # 3. Human Alignment Distribution
        plt.subplot(2, 2, 3)
        human_aligned = results_df['human_aligned']
        labels = ['Below Human', 'Human+ Level']
        colors = ['lightcoral', 'lightgreen']
        sizes = [len(results_df) - human_aligned.sum(), human_aligned.sum()]
        
        plt.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
        plt.title(f'Human Alignment Distribution\n({human_aligned.sum()}/{len(results_df)} ≥ Human Baseline)')
        
        # 4. Cost Breakdown
        plt.subplot(2, 2, 4)
        avg_salary = results_df['salary_cost'].mean()
        avg_time = results_df['time_cost'].mean()
        avg_org = results_df['organizational_cost'].mean()
        
        categories = ['Salary Impact', 'Correction Time', 'Organizational']
        values = [avg_salary, avg_time, avg_org]
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        bars = plt.bar(categories, values, color=colors, alpha=0.8)
        plt.ylabel('Average Cost ($)')
        plt.title('Average Cost Components\nper Classification')
        plt.xticks(rotation=45)
        
        # Add value labels on bars
        for bar, value in zip(bars, values):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                    f'${value:,.0f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(Config.VISUALIZATION_PATH, dpi=300, bbox_inches='tight')
        print(f"💾 Visualization saved to: {Config.VISUALIZATION_PATH}")
        plt.show()
        
    def generate_executive_summary(self, results_df: pd.DataFrame) -> str:
        """Generate executive summary of evaluation results"""
        
        # Calculate key metrics
        total_records = len(results_df)
        avg_likelihood = results_df['likelihood_score'].mean()
        human_aligned_count = results_df['human_aligned'].sum()
        high_performers = (results_df['likelihood_score'] >= 4.0).sum()
        critical_errors = (results_df['error_severity'] > 0.7).sum()
        total_cost = results_df['total_error_cost'].sum()
        avg_correction_time = results_df['correction_hours'].mean()
        
        # ROI calculation
        total_roi = results_df['roi_savings'].sum()
        
        summary = f"""
================================================================================
MNPS JOB CLASSIFICATION LIKELIHOOD ASSESSMENT - EXECUTIVE SUMMARY
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Run Folder: {RUN_TIMESTAMP}
================================================================================

OVERALL PERFORMANCE
----------------------------------------
Total Classifications Evaluated: {total_records}
Average Likelihood Score: {avg_likelihood:.2f}/5.00
Human-Level Performance (≥2.5): {human_aligned_count}/{total_records} ({human_aligned_count/total_records*100:.1f}%)
High Performance (≥4.0): {high_performers}/{total_records} ({high_performers/total_records*100:.1f}%)
Below Human Baseline: {total_records - human_aligned_count}/{total_records} ({(total_records - human_aligned_count)/total_records*100:.1f}%)

ERROR ANALYSIS
----------------------------------------
Average Error Severity: {results_df['error_severity'].mean():.3f}
Critical Errors (>0.7): {critical_errors}
Total Annual Error Cost: ${total_cost:,.2f}
Average Correction Time: {avg_correction_time:.1f} hours
Total Potential ROI: ${total_roi:,.2f}

PRIORITY RECOMMENDATIONS
================================================================================
1. IMMEDIATE ACTIONS:
   • Review {critical_errors} critical errors immediately
   • Focus on {(results_df['likelihood_score'] < 2.0).sum()} classifications below human baseline
   • Prioritize corrections for highest cost errors first

2. SYSTEM IMPROVEMENTS:
   • Implement confidence thresholds for automatic review triggers
   • Consider human expert validation for borderline cases (2.0-3.0 likelihood)
   • Track correction effectiveness over time

3. COST OPTIMIZATION:
   • Total error cost: ${total_cost:,.2f}
   • Potential savings from corrections: ${total_roi:,.2f}
   • Average cost per error: ${total_cost/total_records:,.2f}
   • Focus on high-impact, low-correction-cost classifications first

OUTPUT FILES LOCATION
================================================================================
All results saved to: {CURRENT_RUN_PATH}
  - likelihood_evaluation_results.csv
  - executive_summary_report.txt
  - likelihood_analysis_plots.png
  - run_configuration.json
"""
        return summary

In [ ]:
#===============================================================
# MAIN EXECUTION
#===============================================================

def main():
    """Main execution function with comprehensive error handling"""
    
    print("🚀 Starting MNPS Job Classification Likelihood Assessment...")
    
    try:
        # Initialize pipeline
        pipeline = MNPSEvaluationPipeline()
        visualizer = EvaluationVisualizer()
        
        print("✅ Pipeline initialized successfully")
        
        # Run evaluation
        print("\n🔍 Running evaluation pipeline...")
        results_df = pipeline.run_evaluation(original_df, predicted_df)
        
        print(f"✅ Evaluation complete - processed {len(results_df)} classifications")
        
        # Generate visualizations
        print("\n📊 Generating visualizations...")
        visualizer.create_comprehensive_dashboard(results_df)
        
        # Display executive summary
        summary = visualizer.generate_executive_summary(results_df)
        print("\n" + "="*80)
        print(summary)
        print("="*80)
        
        # Save results to Google Drive
        print("\n💾 Saving results to Google Drive...")
        results_df.to_csv(Config.RESULTS_OUTPUT_PATH, index=False)
        print(f"✅ Results saved to: {Config.RESULTS_OUTPUT_PATH}")
        
        # Save executive summary
        with open(Config.EXECUTIVE_SUMMARY_PATH, 'w') as f:
            f.write(summary)
        print(f"✅ Executive summary saved to: {Config.EXECUTIVE_SUMMARY_PATH}")
        
        # Save configuration log
        config_log = {
            'run_timestamp': RUN_TIMESTAMP,
            'input_files': {
                'sample_jds': SAMPLE_JDS_PATH,
                'classifications': JOB_CLASSIFICATIONS_PATH
            },
            'output_files': {
                'results': Config.RESULTS_OUTPUT_PATH,
                'summary': Config.EXECUTIVE_SUMMARY_PATH,
                'visualization': Config.VISUALIZATION_PATH
            },
            'parameters': {
                'human_baseline': Config.HUMAN_BASELINE,
                'severity_thresholds': {
                    'minor': Config.SEVERITY_MINOR,
                    'major': Config.SEVERITY_MAJOR,
                    'critical': Config.SEVERITY_CRITICAL
                },
                'cost_parameters': {
                    'hourly_rate': Config.HOURLY_RATE,
                    'complexity_factor': Config.COMPLEXITY_FACTOR,
                    'strategic_weight_base': Config.STRATEGIC_WEIGHT_BASE
                }
            },
            'results_summary': {
                'total_records': len(results_df),
                'avg_likelihood': float(results_df['likelihood_score'].mean()),
                'human_aligned_count': int(results_df['human_aligned'].sum()),
                'critical_errors': int((results_df['error_severity'] > 0.7).sum())
            }
        }
        
        with open(Config.CONFIG_LOG_PATH, 'w') as f:
            json.dump(config_log, f, indent=2)
        print(f"✅ Configuration log saved to: {Config.CONFIG_LOG_PATH}")
        
        print("\n✅ All tasks completed successfully!")
        print(f"\n📁 All results available in: {CURRENT_RUN_PATH}")
        
        return results_df
        
    except Exception as e:
        print(f"\n❌ ERROR during execution: {e}")
        import traceback
        traceback.print_exc()
        raise

# Execute the main pipeline
if __name__ == "__main__":
    results = main()

## Additional Analysis and Insights

After running the main evaluation, you can use the cells below for additional analysis and exploration of your results.

In [ ]:
# View top performing classifications
print("\n🏆 TOP 10 PERFORMING CLASSIFICATIONS:")
print("="*80)
top_performers = results.nlargest(10, 'likelihood_score')[[
    'job_title_original', 'new_job_title', 'likelihood_score', 
    'confidence_category', 'major_role_group'
]]
print(top_performers.to_string(index=False))
print(f"\n✅ These {len(top_performers)} classifications show strong alignment with expected outcomes")

In [ ]:
# View classifications requiring immediate review
print("\n🚨 CLASSIFICATIONS REQUIRING IMMEDIATE REVIEW:")
print("="*80)
critical_reviews = results[results['error_severity'] > Config.SEVERITY_MAJOR][[
    'job_title_original', 'new_job_title', 'error_severity', 
    'total_error_cost', 'recommendation'
]].sort_values('error_severity', ascending=False)

if len(critical_reviews) > 0:
    print(critical_reviews.to_string(index=False))
    print(f"\n⚠️  Total: {len(critical_reviews)} classifications need immediate attention")
    print(f"💰 Total cost impact: ${critical_reviews['total_error_cost'].sum():,.2f}")
else:
    print("✅ No critical errors found! All classifications are within acceptable ranges.")

In [ ]:
# Summary statistics
print("\n📈 DETAILED STATISTICS:")
print("="*80)
print(f"\n📊 Likelihood Score Statistics:")
print(results['likelihood_score'].describe())

print(f"\n⚠️  Error Severity Statistics:")
print(results['error_severity'].describe())

print(f"\n💰 Total Error Cost Statistics:")
print(results['total_error_cost'].describe())

print(f"\n⏱️  Correction Hours Statistics:")
print(results['correction_hours'].describe())

In [ ]:
# Performance breakdown by model (if model_used column exists)
if 'model_used' in results.columns and results['model_used'].notna().any():
    print("\n🤖 PERFORMANCE BREAKDOWN BY MODEL:")
    print("="*80)
    model_stats = results.groupby('model_used').agg({
        'likelihood_score': ['count', 'mean', 'std'],
        'error_severity': 'mean',
        'total_error_cost': 'sum',
        'human_aligned': 'sum'
    }).round(2)
    print(model_stats)
else:
    print("\n⚠️  Model information not available in results")

In [ ]:
# Performance breakdown by major role group
if 'major_role_group' in results.columns:
    print("\n📋 PERFORMANCE BREAKDOWN BY MAJOR ROLE GROUP:")
    print("="*80)
    role_stats = results.groupby('major_role_group').agg({
        'likelihood_score': ['count', 'mean'],
        'error_severity': 'mean',
        'human_aligned': lambda x: f"{x.sum()}/{len(x)}"
    }).round(2)
    role_stats.columns = ['Count', 'Avg Likelihood', 'Avg Severity', 'Human Aligned']
    print(role_stats.sort_values('Avg Likelihood', ascending=False))

In [ ]:
# Export filtered datasets for specific analysis
print("\n💾 EXPORTING FILTERED DATASETS...")
print("="*80)

# High performers
high_performers_path = os.path.join(CURRENT_RUN_PATH, "high_performers.csv")
high_performers_df = results[results['likelihood_score'] >= 4.0]
high_performers_df.to_csv(high_performers_path, index=False)
print(f"✅ High performers exported: {len(high_performers_df)} records")
print(f"   Saved to: {high_performers_path}")

# Needs review
needs_review_path = os.path.join(CURRENT_RUN_PATH, "needs_review.csv")
needs_review_df = results[results['error_severity'] > Config.SEVERITY_MINOR]
needs_review_df.to_csv(needs_review_path, index=False)
print(f"✅ Needs review exported: {len(needs_review_df)} records")
print(f"   Saved to: {needs_review_path}")

# Below human baseline
below_baseline_path = os.path.join(CURRENT_RUN_PATH, "below_human_baseline.csv")
below_baseline_df = results[results['human_aligned'] == False]
below_baseline_df.to_csv(below_baseline_path, index=False)
print(f"✅ Below baseline exported: {len(below_baseline_df)} records")
print(f"   Saved to: {below_baseline_path}")

print(f"\n📁 All filtered datasets saved to: {CURRENT_RUN_PATH}")

In [ ]:
# Quick access to results folder in Google Drive
print("\n📂 QUICK ACCESS TO RESULTS:")
print("="*80)
print(f"Run Timestamp: {RUN_TIMESTAMP}")
print(f"Full Path: {CURRENT_RUN_PATH}")
print(f"\nTo access in Google Drive:")
print(f"  Navigate to: My Drive > Likelihood Assessment System > Run Results > {RUN_TIMESTAMP}")
print(f"\n📊 Generated Files:")
for filename in os.listdir(CURRENT_RUN_PATH):
    filepath = os.path.join(CURRENT_RUN_PATH, filename)
    filesize = os.path.getsize(filepath) / 1024  # Size in KB
    print(f"  - {filename} ({filesize:.1f} KB)")

---
## ✅ Evaluation Complete!

All results have been saved to your Google Drive in a timestamped folder for easy tracking.

### Next Steps:
1. Review the executive summary and visualizations
2. Examine classifications flagged for immediate review
3. Compare results across different runs using the timestamped folders
4. Export specific subsets for detailed analysis

### Questions or Issues?
Contact the MNPS HR Analytics team for support.

---